# Project 10: what share of Paramaribo's roads are paved?

**The question.** What share of the road segments in the district of Paramaribo are paved? If a survey team checked only a random sample of segments, how close would its estimate be to the full count, and how wide should the interval it reports be?

**The data.** Every road segment in the district of Paramaribo in OpenStreetMap, downloaded through the Overpass API on 2026-11-06. Map data © OpenStreetMap contributors, under the Open Database License. OpenStreetMap records a surface only where a mapper added one, so in this project "paved share" means the share among segments that carry a surface tag.

**Your team.** Four roles rotate every week: builder, data checker, AI checker and presenter. This week the data checker owns the question of which segments the answer describes.

**What you hand in on Friday.** This notebook with every CHECK passing, one chart of estimates and intervals, a note of about 200 words for the road planners at the ministry responsible for public works, your AI log, and one extra interval from a sample of 100 segments drawn with your team number as the seed.

## Load the data

This cell is given. It reads the table from the course folder, or downloads it from the Overpass API if the file is not there. Each row is one road segment: a piece of road between two points where a mapper split it. The columns are the OpenStreetMap id, the road class (`highway`), the `surface` tag, `paved` (1 for paved, 0 for unpaved, empty when there is no surface tag) and the segment's length in metres.

In [ ]:
import json, math, urllib.parse, urllib.request
import numpy as np
import pandas as pd

QUERY = ('[out:json][timeout:180];'
         'area["ISO3166-2"="SR-PM"]->.paramaribo;'
         'way["highway"~"^(primary|secondary|tertiary|unclassified|residential|living_street|primary_link|secondary_link|tertiary_link)$"](area.paramaribo);'
         'out tags geom;')
# the paved and unpaved groups of the surface tag, from the OpenStreetMap wiki (Key:surface)
PAVED = ["paved", "asphalt", "chipseal", "concrete", "concrete:lanes", "concrete:plates", "paving_stones",
         "paving_stones:lanes", "grass_paver", "sett", "unhewn_cobblestone", "cobblestone", "bricks", "metal",
         "metal_grid", "wood", "stepping_stones", "tiles", "fibre_reinforced_polymer_grate", "resin_bound"]
UNPAVED = ["unpaved", "compacted", "fine_gravel", "gravel", "shells", "rock", "pebblestone", "ground", "dirt",
           "earth", "mud", "laterite", "grass", "sand", "woodchips", "snow", "ice", "salt", "clay"]


def metres(points):
    # length along the points of a segment, on a sphere of radius 6,371 km
    d = 0.0
    for a, b in zip(points, points[1:]):
        p1, p2, dl = math.radians(a["lat"]), math.radians(b["lat"]), math.radians(b["lon"] - a["lon"])
        h = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
        d += 2 * 6371000 * math.asin(math.sqrt(h))
    return d


try:
    roads = pd.read_csv("https://santerre.ai/bigi-data/data/week10-paramaribo-roads.csv")
except FileNotFoundError:
    # the live map changes as people edit it, so your numbers may differ a little from the course file
    req = urllib.request.Request("https://overpass-api.de/api/interpreter",
                                 data=urllib.parse.urlencode({"data": QUERY}).encode(),
                                 headers={"User-Agent": "BiGiData-course/1.0"})
    ways = json.load(urllib.request.urlopen(req, timeout=240))["elements"]
    rows = []
    for way in ways:
        s = way["tags"].get("surface")
        rows.append({"osm_id": way["id"], "highway": way["tags"]["highway"], "surface": s,
                     "paved": 1.0 if s in PAVED else 0.0 if s in UNPAVED else np.nan,
                     "length_m": round(metres(way["geometry"]), 1)})
    roads = pd.DataFrame(rows)
roads

## TODO 1: describe the table

Print the number of segments and the number in each road class. Store in `n_tagged` the number of segments that carry a surface tag.

In [ ]:
# TODO 1
n_tagged = None

In [ ]:
# CHECK 1
assert n_tagged is not None, "n_tagged is still empty"
assert n_tagged == roads["surface"].notna().sum()
print("CHECK 1 passed:", n_tagged, "of", len(roads), "segments carry a surface tag")

## TODO 2: who got a tag?

Store in `coverage` the share of segments with a surface tag in each road class, as a pandas Series indexed by class. Then print the paved share among the tagged segments of each class.

In a text cell, answer three questions. Among the classes with at least 100 segments, which is tagged least often? Are its tagged segments more or less often paved than the rest? What does that suggest about the untagged segments?

In [ ]:
# TODO 2
coverage = None

In [ ]:
# CHECK 2
expected = roads["surface"].notna().groupby(roads["highway"]).mean()
assert coverage is not None, "coverage is still empty"
assert set(coverage.index) == set(expected.index)
assert all(abs(coverage[k] - expected[k]) < 1e-9 for k in expected.index)
big = roads["highway"].value_counts()
print("CHECK 2 passed:", {k: round(float(expected[k]), 3) for k in big[big >= 100].index})

## TODO 3: the full count

From here on, the tagged segments are your population. Store them in a table called `tagged`, and store their paved share in `true_share`. In real survey work nobody knows this number. Here you know it, so you can judge every sample against it.

In [ ]:
# TODO 3
tagged = None
true_share = None

In [ ]:
# CHECK 3
assert tagged is not None and true_share is not None, "tagged or true_share is still empty"
assert len(tagged) == roads["paved"].notna().sum()
assert abs(true_share - roads["paved"].mean()) < 1e-9
print("CHECK 3 passed: the full count covers", len(tagged), "tagged segments")

## TODO 4: one survey of 100 segments

Draw a random sample of 100 tagged segments with `tagged.sample(n=100, random_state=10)` and store it in `sample_100`. Store its paved share in `est_100`.

Then build a 95% bootstrap interval: make a generator with `np.random.default_rng(10)`, draw 2,000 resamples of the 100 `paved` values with replacement, and store the 2.5th and 97.5th percentiles of their shares in `lo_100` and `hi_100`. Print whether the interval holds `true_share`.

In [ ]:
# TODO 4
sample_100 = None
est_100 = None
lo_100, hi_100 = None, None

In [ ]:
# CHECK 4
assert sample_100 is not None and est_100 is not None and lo_100 is not None, "a TODO 4 variable is still empty"
assert len(sample_100) == 100 and sample_100["osm_id"].is_unique
assert set(sample_100["osm_id"]) == set(tagged.sample(n=100, random_state=10)["osm_id"]), "draw the sample with random_state=10"
assert abs(est_100 - sample_100["paved"].mean()) < 1e-9
se_100 = np.sqrt(est_100 * (1 - est_100) / 100)
assert lo_100 < est_100 < hi_100, "the interval must contain the estimate"
assert 0.6 * 2 * 1.96 * se_100 < hi_100 - lo_100 < 1.6 * 2 * 1.96 * se_100, "the width is far from what the standard error predicts"
print("CHECK 4 passed: interval width", round(hi_100 - lo_100, 3))

## TODO 5: samples of different sizes

Repeat TODO 4 for samples of 25, 50, 100, 200 and 400 segments. Draw each with `random_state` set to its size, so the samples are independent of each other, and bootstrap each with a new `np.random.default_rng(10)`. Store a table in `results` with one row per size and the columns `n`, `estimate`, `lo`, `hi`, `width` and `holds_full_count`.

Draw one chart: the estimate for each size with its interval as an error bar, and the full count as a dashed line. Give it a title and label both axes.

In [ ]:
# TODO 5
results = None

In [ ]:
# CHECK 5
assert results is not None, "results is still empty"
assert list(results["n"]) == [25, 50, 100, 200, 400]
assert all(results["lo"] <= results["estimate"]) and all(results["estimate"] <= results["hi"])
assert results["width"].iloc[-1] < results["width"].iloc[:3].min(), "the interval should narrow as the sample grows"
print("CHECK 5 passed: widths", [round(v, 3) for v in results["width"]])

## TODO 6: does 95 percent hold?

One interval either holds the full count or misses it. To test the method, repeat the survey 200 times. Make a generator with `np.random.default_rng(10)`; for each repetition, draw 100 segments without replacement from the `paved` values of `tagged`, build a bootstrap interval from 1,000 resamples, and record whether it holds `true_share`. Store the share of the 200 intervals that hold it in `coverage_rate`.

In [ ]:
# TODO 6
coverage_rate = None

In [ ]:
# CHECK 6
assert coverage_rate is not None, "coverage_rate is still empty"
assert 0.85 <= coverage_rate <= 1.0, "a 95% method should hold the full count in most of the 200 surveys"
print("CHECK 6 passed: coverage", round(coverage_rate, 3))

## TODO 7: the AI check

First ask your AI assistant, without giving it any data: *"What share of the roads in Paramaribo are paved?"* Write its answer in your AI log. Did it give one number or a range? Did it name a source and a date?

Then print `list(sample_100["paved"].astype(int))`, paste the 100 values into the chat, and ask: *"These are 100 randomly chosen road segments, 1 for paved and 0 for unpaved. Give me a 95% interval for the paved share of all segments."* Record the interval and the formula it used.

In the cell below, store two intervals as (low, high) pairs. In `sd_interval`, use `est_100` plus and minus 1.96 times the standard deviation of the 100 values. In `se_interval`, use 1.96 times the standard error instead. Print both next to your bootstrap interval, and write in your log which one the AI's answer resembles.

In [ ]:
# TODO 7
sd_interval = None
se_interval = None

In [ ]:
# CHECK 7
assert sd_interval is not None and se_interval is not None, "sd_interval or se_interval is still empty"
sd_width = sd_interval[1] - sd_interval[0]
se_width = se_interval[1] - se_interval[0]
assert abs(sd_width / se_width - 10) < 0.01, "the standard error is the standard deviation divided by the square root of 100"
assert abs(se_width - (hi_100 - lo_100)) < 0.3 * (hi_100 - lo_100), "the standard-error interval should be close to the bootstrap interval"
print("CHECK 7 passed: the standard-deviation interval is", round(sd_width / se_width, 1), "times as wide as the standard-error interval")

## TODO 8: segments or kilometres?

A segment is a unit of the map, not of the road network: a long avenue can be ten segments and a side street one. A road planner thinks in kilometres. Store in `length_share` the share of the tagged road length that is paved, and compare it with `true_share`. In a text cell, say which of the two answers the planner's question, and why they differ.

In [ ]:
# TODO 8
length_share = None

In [ ]:
# CHECK 8
assert length_share is not None, "length_share is still empty"
expected = (tagged["paved"] * tagged["length_m"]).sum() / tagged["length_m"].sum()
assert abs(length_share - expected) < 1e-6
print("CHECK 8 passed: the length share differs from the segment share by", round(abs(length_share - true_share), 3))

## TODO 9: the note for the road planners

In a text cell below, write about 200 words for the road planners at the ministry responsible for public works. They decide which streets to survey and which to pave first, so give them what a planner can act on.

- The estimate that a survey of 100 segments would have produced, with its interval, in one honest sentence.
- The full count, and how many segments carry no surface tag, and which road classes they come from.
- How many segments a survey team would need to visit to know the paved share to within 5 percentage points either way, from your TODO 5 table or the formula in the skills notebook.
- The source and its date, and one thing the interval cannot tell the planners.